<p align="center">
  <img src="https://www.copernicuslac-chile.eu/wp-content/uploads/2024/05/logo_copernicuslac_chile.svg" alt="Copernicus LAC Chile" width="1200"/>
</p>

# Taller LCLU — IDE Uruguay 2026
## Comparación de fuentes y cambio temporal de coberturas

Este notebook se organiza en dos partes:

1. **Comparación entre fuentes:** CopernicusLAC Chile y MVOT para un año nominal común.
2. **Evaluación de cambio temporal:** permanencia, pérdida y ganancia de la clase seleccionada entre Coplac 2021 y Coplac 2022.

La comparación entre fuentes no debe interpretarse automáticamente como cambio temporal: sus diferencias pueden deberse a la leyenda, resolución espacial, unidad mínima cartografiable, fecha exacta, método de clasificación o reglas de generalización.

## Pregunta guía

¿Cómo cambia la superficie y la distribución de una cobertura cuando usamos distintas fuentes para el mismo año?

## 1. Importación y configuración

In [ ]:
from pathlib import Path
import sys
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import BoundaryNorm, ListedColormap

SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    candidate
    for candidate in SEARCH_ROOTS
    if (candidate / "data" / "rasters_CyU").is_dir()
 )
PREPROCESSED_TIF_DIR = REPO_ROOT / "data" / "rasters_CyU"
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from lulc_comparison import build_harmonized_crosswalk, summarize_harmonized_raster

def load_processed_raster(filename):
    path = PREPROCESSED_TIF_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"No existe el GeoTIFF procesado: {path}")
    with rasterio.open(path) as dataset:
        return dataset.read(1, out_dtype="uint8"), dataset.transform, dataset.crs

def pixel_area_from_transform(transform, crs):
    if crs is None or not crs.is_projected:
        raise ValueError("El GeoTIFF procesado debe usar un CRS proyectado.")
    return abs(transform.a * transform.e - transform.b * transform.d)

In [ ]:
ANALYSIS_YEAR = 2022
TARGET_CRS = "EPSG:32721"
SOURCE_A_LABEL = "CopernicusLAC Chile"
SOURCE_B_LABEL = "MVOT"
SHOW_PLOTS = True
OUTPUT_DIR = REPO_ROOT / "output" / f"lulc_{ANALYSIS_YEAR}"
PRINTS_DIR = OUTPUT_DIR / "prints"


In [ ]:
CLASS_HARMONIZATION = {
    "superficies_artificiales": {"source_a - coplac": [4096], "source_b - mvot": [5, 8]},
    "cultivo": {"source_a - coplac": [8192], "source_b - mvot": [9, 10, 13]},
    "pastizales": {"source_a - coplac": [12288], "source_b - mvot": [12]},
    "areas_cobertura_arborea": {"source_a - coplac": [16384], "source_b - mvot": [7, 11]},
    "areas_cobertura_arbustiva": {"source_a - coplac": [20480], "source_b - mvot": [3]},
    "vegetacion_herbacea_acuatica": {"source_a - coplac": [24576], "source_b - mvot": [6]},
    "manglares": {"source_a - coplac": [28672], "source_b - mvot": []},
    "vegetacion_escasa": {"source_a - coplac": [32768], "source_b - mvot": []},
    "suelo_desnudo": {"source_a - coplac": [36864], "source_b - mvot": [4]},
    "nieve_y_glaciares": {"source_a - coplac": [40960], "source_b - mvot": []},
    "cuerpo_de_agua": {"source_a - coplac": [45056], "source_b - mvot": [1, 2]},
}
SOURCE_A_KEY = "source_a - coplac"
SOURCE_B_KEY = "source_b - mvot"

## 2. Carga de inputs preprocesados

El notebook no carga rásteres crudos ni ejecuta mosaicos, recortes o reproyecciones. Los GeoTIFF preprocesados se encuentran en `data/tifs` y se cargan directamente para el análisis.

In [ ]:
harmonized_a, transform_a, crs_a = load_processed_raster(
    "nominal_coplac_2022_homologado.tif"
 )
harmonized_b, transform_b, crs_b = load_processed_raster(
    "nominal_mvot_2021_2022_homologado.tif"
 )
valid_a, _, _ = load_processed_raster("nominal_coplac_2022_validez.tif")
valid_b, _, _ = load_processed_raster("nominal_mvot_2021_2022_validez.tif")
valid_a = valid_a.astype(bool)
valid_b = valid_b.astype(bool)
if harmonized_a.shape != harmonized_b.shape or harmonized_a.shape != valid_a.shape or harmonized_b.shape != valid_b.shape:
    raise ValueError("Los GeoTIFF procesados no tienen la misma cuadrícula.")
area_m2 = pixel_area_from_transform(transform_a, crs_a)
source_a = {"array": harmonized_a, "valid": valid_a}
source_b = {"array": harmonized_b, "valid": valid_b}
labels = {code: class_name for code, class_name in enumerate(CLASS_HARMONIZATION, start=1)}
metadata = {"source": "draft/output/preprocessed/tifs", "target_crs": TARGET_CRS}
print(f"GeoTIFF procesados cargados: {PREPROCESSED_TIF_DIR}")
print(f"Área de píxel: {area_m2:.2f} m²")
print(f"Cuadrícula: {harmonized_a.shape[1]} x {harmonized_a.shape[0]} píxeles")

## 3. Construcción de los mapas multiclase homologados

In [ ]:
pd.DataFrame({"codigo_homologado": labels.keys(), "clase": labels.values()})

## 4. Superficie por clase y fuente

In [ ]:
table_a = pd.DataFrame(summarize_harmonized_raster(harmonized_a, source_a["valid"], labels, area_m2))
table_a["fuente"] = SOURCE_A_LABEL
table_b = pd.DataFrame(summarize_harmonized_raster(harmonized_b, source_b["valid"], labels, area_m2))
table_b["fuente"] = SOURCE_B_LABEL
area_by_source = pd.concat([table_a, table_b], ignore_index=True)
area_comparison = area_by_source.pivot(index="clase", columns="fuente", values="area_ha").reset_index()
area_valid_a_ha = source_a["valid"].sum() * area_m2 / 10000
area_valid_b_ha = source_b["valid"].sum() * area_m2 / 10000
area_comparison["area_a_km2"] = area_comparison[SOURCE_A_LABEL] / 100
area_comparison["area_b_km2"] = area_comparison[SOURCE_B_LABEL] / 100
area_comparison["diferencia_ha"] = area_comparison[SOURCE_B_LABEL] - area_comparison[SOURCE_A_LABEL]
area_comparison["diferencia_km2"] = area_comparison["area_b_km2"] - area_comparison["area_a_km2"]
area_comparison["participacion_a_pct"] = 100 * area_comparison[SOURCE_A_LABEL] / area_valid_a_ha
area_comparison["participacion_b_pct"] = 100 * area_comparison[SOURCE_B_LABEL] / area_valid_b_ha
area_comparison["diferencia_participacion_pp"] = area_comparison["participacion_b_pct"] - area_comparison["participacion_a_pct"]
area_comparison["cambio_relativo_desde_a_pct"] = 100 * area_comparison["diferencia_ha"] / area_comparison[SOURCE_A_LABEL].replace(0, np.nan)
area_comparison.sort_values("diferencia_km2", key=lambda values: values.abs(), ascending=False)

In [ ]:
PRINTS_DIR.mkdir(parents=True, exist_ok=True)
plot_area = area_by_source.copy()
plot_area["etiqueta"] = plot_area["codigo_homologado"].astype(str) + " — " + plot_area["clase"]
plot_area = plot_area.sort_values("area_ha")
area_pivot = plot_area.pivot(index="etiqueta", columns="fuente", values="area_ha").fillna(0)
figure, axis = plt.subplots(figsize=(11, max(5, 0.42 * len(area_pivot))), dpi=180, constrained_layout=True)
area_pivot.plot(
    kind="barh",
    ax=axis,
    color=["#2f7d32", "#467ee0"],
 )
axis.set_title(f"Comparación de superficie por clase — {ANALYSIS_YEAR}")
axis.set_xlabel("Área (ha)")
axis.set_ylabel("Clase homologada")
axis.grid(axis="x", alpha=0.25)
axis.set_axisbelow(True)
axis.legend(title="Fuente")
figure.savefig(PRINTS_DIR / "01b_superficie_por_clase_fuente.png", dpi=180, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

difference_plot = area_comparison[["clase", "diferencia_km2"]].sort_values("diferencia_km2")
bar_colors = np.where(difference_plot["diferencia_km2"] >= 0, "#1a9850", "#d73027")
figure, axis = plt.subplots(figsize=(10, max(5, 0.42 * len(difference_plot))), dpi=180, constrained_layout=True)
axis.barh(difference_plot["clase"], difference_plot["diferencia_km2"], color=bar_colors)
axis.axvline(0, color="black", linewidth=0.8)
axis.set_title(f"Diferencia de superficie por clase — {SOURCE_B_LABEL} menos {SOURCE_A_LABEL}")
axis.set_xlabel("Diferencia de área (km²)")
axis.set_ylabel("Clase homologada")
axis.grid(axis="x", alpha=0.25)
axis.set_axisbelow(True)
figure.savefig(PRINTS_DIR / "01c_diferencia_superficie_por_clase.png", dpi=180, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

## 5. Mapas comparables y exportación

In [ ]:
land_cover_qml_colors = {
    4096: "#dd4a4a",
    8192: "#f490dd",
    12288: "#f9ec48",
    16384: "#2d8655",
    20480: "#81cc71",
    24576: "#0db59d",
    28672: "#75d3b8",
    32768: "#d8d79e",
    36864: "#8c5e4c",
    40960: "#eaf2fa",
    45056: "#467ee0",
}
coplac_original_code_by_class_code = {
    class_code: CLASS_HARMONIZATION[class_name][SOURCE_A_KEY][0]
    for class_code, class_name in labels.items()
}
coplac_class_colors_by_code = {
    class_code: land_cover_qml_colors[original_code]
    for class_code, original_code in coplac_original_code_by_class_code.items()
}
mvot_class_colors_by_label = {
    "superficies_artificiales": "#c2185b",
    "cultivo": "#c25fba",
    "pastizales": "#F2D33B",
    "areas_cobertura_arborea": "#2e7d32",
    "areas_cobertura_arbustiva": "#00897b",
    "vegetacion_herbacea_acuatica": "#00acc1",
    "manglares": "#1b5e20",
    "vegetacion_escasa": "#8d6e63",
    "suelo_desnudo": "#ef6c00",
    "nieve_y_glaciares": "#90a4ae",
    "cuerpo_de_agua": "#1565c0",
}
mvot_class_colors_by_code = {
    class_code: mvot_class_colors_by_label[class_name]
    for class_code, class_name in labels.items()
}
codes = list(labels)
norm = BoundaryNorm(np.arange(0.5, len(codes) + 1.5), len(codes))
figure, axes = plt.subplots(1, 2, figsize=(17, 7), dpi=180, constrained_layout=True)
for axis, raster, valid, source_label, color_by_code in zip(
    axes,
    (harmonized_a, harmonized_b),
    (source_a["valid"], source_b["valid"]),
    (SOURCE_A_LABEL, SOURCE_B_LABEL),
    (coplac_class_colors_by_code, mvot_class_colors_by_code),
 ):
    cmap = ListedColormap([color_by_code[code] for code in codes])
    display = np.ma.masked_where((~valid) | (raster == 0), raster)
    axis.imshow(display, cmap=cmap, norm=norm, interpolation="nearest", resample=False)
    axis.set_title(source_label)
    axis.set_axis_off()
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=cmap(norm(code)),
            markersize=9,
            label=f"{code}: {labels[code]}",
        )
        for code in codes
    ]
    axis.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8, title=source_label)
figure.suptitle("Clases homologadas: comparación multiclase")
PRINTS_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(PRINTS_DIR / "01_mapas_multiclase_homologados.png", dpi=180, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

## 6. Selección de la clase a analizar

Selecciona una clase homologada en el widget y confirma la elección antes de ejecutar las celdas de comparación espacial o cambio temporal. La clase seleccionada se guarda en `SELECTED_CLASS` y se reutiliza en ambas partes del notebook.

In [ ]:
from IPython.display import clear_output, display

CLASS_OPTIONS = list(labels.values())
SELECTED_CLASS = "areas_cobertura_arborea" if "areas_cobertura_arborea" in CLASS_OPTIONS else CLASS_OPTIONS[0]
WIDGETS_AVAILABLE = True

try:
    import ipywidgets as widgets
except ImportError:
    WIDGETS_AVAILABLE = False

if WIDGETS_AVAILABLE:
    class_selector = widgets.Dropdown(
        options=CLASS_OPTIONS,
        value=SELECTED_CLASS,
        description="Clase:",
        layout=widgets.Layout(width="420px"),
    )
    confirm_class_button = widgets.Button(
        description="Confirmar clase",
        button_style="success",
        icon="check",
    )
    class_selection_output = widgets.Output()

    def confirm_class_selection(_):
        global SELECTED_CLASS
        SELECTED_CLASS = class_selector.value
        with class_selection_output:
            clear_output(wait=True)
            print(f"Clase seleccionada: {SELECTED_CLASS}")

    confirm_class_button.on_click(confirm_class_selection)
    display(widgets.VBox([class_selector, confirm_class_button, class_selection_output]))
else:
    print(f"ipywidgets no está disponible. Se usará la clase predeterminada: {SELECTED_CLASS}")

In [ ]:
selected_codes = [code for code, name in labels.items() if name == SELECTED_CLASS]
if not selected_codes:
    raise ValueError(f"Clase no encontrada: {SELECTED_CLASS}. Opciones: {CLASS_OPTIONS}")
selected_code = selected_codes[0]

valid_comparison = source_a["valid"] & source_b["valid"]
class_a = valid_comparison & (harmonized_a == selected_code)
class_b = valid_comparison & (harmonized_b == selected_code)
class_disagreement = valid_comparison & (class_a != class_b)

print(f"Clase seleccionada: {SELECTED_CLASS} (código {selected_code})")
print(f"Píxeles de desacuerdo: {int(class_disagreement.sum()):,}")

## 7. Tres zonas con mayor concentración de diferencias

La cuenca se divide en una cuadrícula regular y se seleccionan las tres ventanas con más diferencias de presencia/ausencia para la clase elegida.

In [ ]:
GRID_ROWS = 4
GRID_COLUMNS = 4
TOP_ZONES = 3

height, width = class_disagreement.shape
valid_area_km2 = valid_comparison.sum() * area_m2 / 1_000_000
total_disagreement_pixels = int(class_disagreement.sum())
class_area_a_km2 = class_a.sum() * area_m2 / 1_000_000
class_area_b_km2 = class_b.sum() * area_m2 / 1_000_000
row_edges = np.linspace(0, height, GRID_ROWS + 1, dtype=int)
column_edges = np.linspace(0, width, GRID_COLUMNS + 1, dtype=int)
zone_scores = []

for row_index in range(GRID_ROWS):
    for column_index in range(GRID_COLUMNS):
        row_start, row_stop = row_edges[row_index], row_edges[row_index + 1]
        column_start, column_stop = column_edges[column_index], column_edges[column_index + 1]
        score = int(class_disagreement[row_start:row_stop, column_start:column_stop].sum())
        if score:
            disagreement_km2 = score * area_m2 / 1_000_000
            zone_scores.append(
                {
                    "row_index": row_index,
                    "column_index": column_index,
                    "row_start": row_start,
                    "row_stop": row_stop,
                    "column_start": column_start,
                    "column_stop": column_stop,
                    "disagreement_pixels": score,
                    "disagreement_km2": disagreement_km2,
                    "disagreement_pct_valid": 100 * disagreement_km2 / valid_area_km2,
                    "disagreement_pct_total": 100 * score / total_disagreement_pixels,
                }
            )

top_zones = sorted(
    zone_scores,
    key=lambda zone: zone["disagreement_pixels"],
    reverse=True,
 )[:TOP_ZONES]
pd.DataFrame(top_zones)

## 8. Exportación de los tres acercamientos

Cada zona seleccionada se representa en una figura de tres paneles y se guarda en `output/multiclase_2022/prints`.

In [ ]:
from matplotlib.patches import Rectangle

class_cmap = ListedColormap(["#f1f3f0", "#2f7d32"])
class_norm = BoundaryNorm([-0.5, 0.5, 1.5], class_cmap.N)
difference_cmap = ListedColormap(["#eeeeee", "#d1495b", "#e07a2d"])
difference_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], difference_cmap.N)
PRINTS_DIR.mkdir(parents=True, exist_ok=True)

for zone_number, zone in enumerate(top_zones, start=1):
    row_slice = slice(zone["row_start"], zone["row_stop"])
    column_slice = slice(zone["column_start"], zone["column_stop"])
    zone_a = class_a[row_slice, column_slice]
    zone_b = class_b[row_slice, column_slice]
    zone_valid = valid_comparison[row_slice, column_slice]
    zone_a = np.ma.masked_where(~zone_valid, zone_a, copy=False)
    zone_b = np.ma.masked_where(~zone_valid, zone_b, copy=False)

    zone_difference = np.full(zone_valid.shape, np.nan)
    zone_difference[zone_valid & ~class_a[row_slice, column_slice] & ~class_b[row_slice, column_slice]] = 0
    zone_difference[zone_valid & class_a[row_slice, column_slice] & ~class_b[row_slice, column_slice]] = 1
    zone_difference[zone_valid & ~class_a[row_slice, column_slice] & class_b[row_slice, column_slice]] = 2
    zone_difference = np.ma.masked_invalid(zone_difference)

    figure, axes = plt.subplots(
        1,
        3,
        figsize=(17, 5.5),
        dpi=180,
        constrained_layout=True,
    )
    for axis, raster, title in zip(
        axes[:2],
        (zone_a, zone_b),
        (SOURCE_A_LABEL, SOURCE_B_LABEL),
    ): 
        axis.imshow(
            raster,
            cmap=class_cmap,
            norm=class_norm,
            interpolation="nearest",
            resample=False,
        )
        axis.set_title(f"{title}: {SELECTED_CLASS}")
        axis.set_axis_off()
    axes[2].imshow(
        zone_difference,
        cmap=difference_cmap,
        norm=difference_norm,
        interpolation="nearest",
        resample=False,
    )
    axes[2].set_title("Coincide / solo CopernicusLAC / solo MVOT")
    axes[2].set_axis_off()
    figure.suptitle(
        f"{SELECTED_CLASS} - zona {zone_number}: {zone['disagreement_km2']:.2f} km² de diferencia"
    )
    output_path = PRINTS_DIR / f"02_zona_{SELECTED_CLASS}_desacuerdos_{zone_number:02d}.png"
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(figure)

overview_figure, overview_axis = plt.subplots(
    figsize=(9, 7),
    dpi=180,
    constrained_layout=True,
 )
overview_class = np.ma.masked_where(~valid_comparison, class_a, copy=False)
overview_axis.imshow(
    overview_class,
    cmap=class_cmap,
    norm=class_norm,
    interpolation="nearest",
    resample=False,
 )
overview_disagreement = np.ma.masked_where(~class_disagreement, class_disagreement, copy=False)
overview_axis.imshow(
    overview_disagreement,
    cmap="Reds",
    alpha=0.85,
    interpolation="nearest",
    resample=False,
 )
zone_colors = ["#1b9e77", "#d95f02", "#7570b3"]
for zone_number, zone in enumerate(top_zones, start=1):
    rectangle = Rectangle(
        (zone["column_start"], zone["row_start"]),
        zone["column_stop"] - zone["column_start"],
        zone["row_stop"] - zone["row_start"],
        fill=False,
        edgecolor=zone_colors[(zone_number - 1) % len(zone_colors)],
        linewidth=2.5,
    )
    overview_axis.add_patch(rectangle)
    overview_axis.text(
        zone["column_start"] + 8,
        zone["row_start"] + 24,
        f"Zona {zone_number}",
        color="white",
        fontsize=10,
        fontweight="bold",
        bbox={"facecolor": rectangle.get_edgecolor(), "alpha": 0.85, "pad": 3},
    )
overview_axis.set_title(f"Mapa de referencia: {SELECTED_CLASS}")
overview_axis.set_axis_off()
overview_figure.savefig(
    PRINTS_DIR / f"02_mapa_referencia_zonas_{SELECTED_CLASS}.png",
    dpi=180,
    bbox_inches="tight",
 )
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(overview_figure)

print(f"Figuras exportadas: {len(top_zones)}")
print(f"Área de la clase en {SOURCE_A_LABEL}: {class_area_a_km2:.2f} km²")
print(f"Área de la clase en {SOURCE_B_LABEL}: {class_area_b_km2:.2f} km²")

for variable_name in (
    "figure", "axes", "axis", "raster", "overview_figure", "overview_axis",
    "overview_class", "overview_disagreement", "zone_a", "zone_b", "zone_valid",
    "zone_difference", "display",
): 
    globals().pop(variable_name, None)
gc.collect()

## 9. Correspondencias entre clases

La correspondencia se calcula sobre toda el área válida común de la cuenca. No se restringe a los tres hotspots seleccionados para las figuras de desacuerdo. La matriz se representa como porcentaje de cada clase de Coplac distribuido entre las clases de MVOT.

In [ ]:
crosswalk = pd.DataFrame(build_harmonized_crosswalk(
    harmonized_a, harmonized_b, source_a["valid"], source_b["valid"], labels, area_m2
))
crosswalk["area_km2"] = crosswalk["area_ha"] / 100
crosswalk.sort_values("area_km2", ascending=False)

In [ ]:
crosswalk_matrix = crosswalk.pivot_table(index="fuente_a", columns="fuente_b", values="area_km2", aggfunc="sum", fill_value=0)
crosswalk_pct = crosswalk_matrix.div(
    crosswalk_matrix.sum(axis=1).replace(0, np.nan),
    axis=0,
 ) * 100
figure, axis = plt.subplots(figsize=(12, 8), dpi=180, constrained_layout=True)
image = axis.imshow(
    crosswalk_pct.values,
    cmap="YlGnBu",
    vmin=0,
    vmax=100,
    interpolation="nearest",
    resample=False,
 )
axis.set_xticks(range(len(crosswalk_pct.columns)), crosswalk_pct.columns, rotation=45, ha="right")
axis.set_yticks(range(len(crosswalk_pct.index)), crosswalk_pct.index)
axis.set_xlabel(f"Clase en {SOURCE_B_LABEL}")
axis.set_ylabel(f"Clase en {SOURCE_A_LABEL}")
axis.set_title("Correspondencia relativa por clase de Coplac")
figure.colorbar(image, ax=axis, label="% de la clase de Coplac")
for row_index in range(crosswalk_pct.shape[0]):
    for column_index in range(crosswalk_pct.shape[1]):
        value = crosswalk_pct.iat[row_index, column_index]
        if value > 0.5:
            axis.text(
                column_index,
                row_index,
                f"{value:.0f}%",
                ha="center",
                va="center",
                color="black" if value < 55 else "white",
                fontsize=8,
            )
figure.savefig(
    PRINTS_DIR / "02_matriz_correspondencias_multiclase_porcentaje.png",
    dpi=180,
    bbox_inches="tight",
 )
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

In [ ]:
crosswalk_plot = crosswalk.sort_values("area_km2", ascending=True).tail(15)
crosswalk_plot["transicion"] = crosswalk_plot["fuente_a"] + " → " + crosswalk_plot["fuente_b"]
figure, axis = plt.subplots(figsize=(11, max(5, 0.38 * len(crosswalk_plot))), dpi=180, constrained_layout=True)
axis.barh(crosswalk_plot["transicion"], crosswalk_plot["area_km2"], color="#467ee0")
axis.set_title("Principales correspondencias espaciales entre fuentes")
axis.set_xlabel("Área correspondiente (km²)")
axis.set_ylabel("Clase Coplac → clase MVOT")
axis.grid(axis="x", alpha=0.25)
axis.set_axisbelow(True)
figure.savefig(PRINTS_DIR / "02b_principales_correspondencias_espaciales.png", dpi=180, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

## Parte 2. Cambio temporal de la clase seleccionada

Esta sección compara Coplac 2021 y Coplac 2022 sobre la misma cuenca y con la misma clase homologada seleccionada en la Parte 1. Los resultados distinguen permanencia, pérdida, ganancia y superficie sin la clase seleccionada en ambos años.

In [ ]:
# Liberar los rásteres nominales antes de cargar los temporales.
for variable_name in (
    "harmonized_a", "harmonized_b", "valid_a", "valid_b",
    "source_a", "source_b", "class_a", "class_b", "class_disagreement",
): 
    globals().pop(variable_name, None)
gc.collect()

CHANGE_YEAR_A = 2021
CHANGE_YEAR_B = 2022
CHANGE_SOURCE_LABEL = "CopernicusLAC Chile"
CHANGE_CLASS_NAME = SELECTED_CLASS
change_harmonized_a, change_transform_a, change_crs_a = load_processed_raster(
    "temporal_coplac_2021_homologado.tif"
 )
change_harmonized_b, change_transform_b, change_crs_b = load_processed_raster(
    "temporal_coplac_2022_homologado.tif"
 )
change_valid_a, _, _ = load_processed_raster("temporal_coplac_2021_validez.tif")
change_valid_b, _, _ = load_processed_raster("temporal_coplac_2022_validez.tif")
change_source_a = {"array": change_harmonized_a, "valid": change_valid_a.astype(bool)}
change_source_b = {"array": change_harmonized_b, "valid": change_valid_b.astype(bool)}
change_area_m2 = pixel_area_from_transform(change_transform_a, change_crs_a)
change_labels = {code: class_name for code, class_name in enumerate(CLASS_HARMONIZATION, start=1)}
change_codes = [code for code, name in change_labels.items() if name == CHANGE_CLASS_NAME]
if not change_codes:
    raise ValueError(f"Clase no encontrada para el cambio temporal: {CHANGE_CLASS_NAME}")
change_code = change_codes[0]

change_valid = change_source_a["valid"] & change_source_b["valid"]
change_mask_a = change_valid & (change_harmonized_a == change_code)
change_mask_b = change_valid & (change_harmonized_b == change_code)
print(f"GeoTIFF temporales cargados: {PREPROCESSED_TIF_DIR}")
print(f"Cambio temporal: {CHANGE_YEAR_A} a {CHANGE_YEAR_B}")
print(f"Clase analizada: {CHANGE_CLASS_NAME} (código {change_code})")
print(f"Área válida común: {change_valid.sum() * change_area_m2 / 1_000_000:.2f} km²")

In [ ]:
change_both = change_valid & change_mask_a & change_mask_b
change_loss = change_valid & change_mask_a & ~change_mask_b
change_gain = change_valid & ~change_mask_a & change_mask_b
change_neither = change_valid & ~change_mask_a & ~change_mask_b

change_area_km2 = {
    "permanencia": change_both.sum() * change_area_m2 / 1_000_000,
    "perdida": change_loss.sum() * change_area_m2 / 1_000_000,
    "ganancia": change_gain.sum() * change_area_m2 / 1_000_000,
    "resto": change_neither.sum() * change_area_m2 / 1_000_000,
}
change_valid_area_km2 = change_valid.sum() * change_area_m2 / 1_000_000
change_area_a_km2 = change_mask_a.sum() * change_area_m2 / 1_000_000
change_area_b_km2 = change_mask_b.sum() * change_area_m2 / 1_000_000

change_summary = pd.DataFrame(
    [
        {
            "clase": CHANGE_CLASS_NAME,
            "periodo": f"{CHANGE_YEAR_A}-{CHANGE_YEAR_B}",
            f"area_{CHANGE_YEAR_A}_km2": change_area_a_km2,
            f"area_{CHANGE_YEAR_B}_km2": change_area_b_km2,
            f"porcentaje_{CHANGE_YEAR_A}_area_valida": 100 * change_area_a_km2 / change_valid_area_km2 if change_valid_area_km2 else np.nan,
            f"porcentaje_{CHANGE_YEAR_B}_area_valida": 100 * change_area_b_km2 / change_valid_area_km2 if change_valid_area_km2 else np.nan,
            "cambio_neto_km2": change_area_b_km2 - change_area_a_km2,
            "permanencia_km2": change_area_km2["permanencia"],
            "perdida_km2": change_area_km2["perdida"],
            "ganancia_km2": change_area_km2["ganancia"],
            "permanencia_pct_area_valida": 100 * change_area_km2["permanencia"] / change_valid_area_km2 if change_valid_area_km2 else np.nan,
            "perdida_pct_area_valida": 100 * change_area_km2["perdida"] / change_valid_area_km2 if change_valid_area_km2 else np.nan,
            "ganancia_pct_area_valida": 100 * change_area_km2["ganancia"] / change_valid_area_km2 if change_valid_area_km2 else np.nan,
            "tasa_perdida_pct": 100 * change_area_km2["perdida"] / change_area_a_km2 if change_area_a_km2 else np.nan,
            "tasa_ganancia_pct": 100 * change_area_km2["ganancia"] / change_area_b_km2 if change_area_b_km2 else np.nan,
            "cambio_neto_pct_area_valida": 100 * (change_area_b_km2 - change_area_a_km2) / change_valid_area_km2 if change_valid_area_km2 else np.nan,
        }
    ]
 )
change_summary.round(3)

In [ ]:
change_plot = pd.Series(change_area_km2, name="area_km2").sort_values()
figure, axis = plt.subplots(figsize=(9, 5), dpi=180, constrained_layout=True)
axis.barh(
    change_plot.index,
    change_plot.values,
    color=["#d9d9d9", "#d73027", "#1a9850", "#ffeb3b"],
 )
axis.set_title(f"Cambio temporal de {CHANGE_CLASS_NAME}: {CHANGE_YEAR_A}-{CHANGE_YEAR_B}")
axis.set_xlabel("Área (km²)")
axis.set_ylabel("Categoría")
axis.grid(axis="x", alpha=0.25)
axis.set_axisbelow(True)
figure.savefig(
    PRINTS_DIR / f"03b_resumen_cambio_{CHANGE_CLASS_NAME}_{CHANGE_YEAR_A}_{CHANGE_YEAR_B}.png",
    dpi=180,
    bbox_inches="tight",
 )
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

### Mapas y exportación del cambio temporal

In [ ]:
from matplotlib.patches import Patch

change_cmap = ListedColormap(["#eeeeee", "#bdbdbd", "#d1495b", "#e07a2d", "#467ee0"])
change_labels = [
    "sin datos",
    "resto",
    "permanencia",
    "pérdida",
    "ganancia",
]
change_figure, change_axes = plt.subplots(3, 1, figsize=(8, 14), dpi=180, constrained_layout=True)

for axis, mask, title in zip(
    change_axes[:2],
    (change_mask_a, change_mask_b),
    (str(CHANGE_YEAR_A), str(CHANGE_YEAR_B)),
 ):
    display = np.ma.masked_where(~change_valid, mask, copy=False)
    axis.imshow(
        display,
        cmap=ListedColormap(["#f1f3f0", "#bdbdbd"]),
        norm=BoundaryNorm([-0.5, 0.5, 1.5], 2),
        interpolation="nearest",
        resample=False,
    )
    axis.set_title(f"{CHANGE_CLASS_NAME}: {title}")
    axis.set_axis_off()

change_map = np.zeros(change_valid.shape, dtype=np.uint8)
change_map[change_neither] = 1
change_map[change_both] = 2
change_map[change_loss] = 3
change_map[change_gain] = 4
masked_change_map = np.ma.masked_where(~change_valid, change_map, copy=False)
change_axes[2].imshow(
    masked_change_map,
    cmap=change_cmap,
    norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], 5),
    interpolation="nearest",
    resample=False,
 )
change_axes[2].set_title("Permanencia, pérdida y ganancia")
change_axes[2].set_axis_off()
change_handles = [
    Patch(color=color, label=label)
    for color, label in zip(change_cmap.colors, change_labels)
]
change_axes[2].legend(handles=change_handles, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
change_figure.suptitle(
    f"Cambio temporal de {CHANGE_CLASS_NAME}: {CHANGE_YEAR_A}-{CHANGE_YEAR_B}"
 )
change_figure.savefig(
    PRINTS_DIR / f"03_cambio_{CHANGE_CLASS_NAME}_{CHANGE_YEAR_A}_{CHANGE_YEAR_B}.png",
    dpi=180,
    bbox_inches="tight",
 )
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(change_figure)

change_summary.to_csv(
    OUTPUT_DIR / f"cambio_{CHANGE_CLASS_NAME}_{CHANGE_YEAR_A}_{CHANGE_YEAR_B}.csv",
    index=False,
 )
print(f"Resumen temporal guardado en: {OUTPUT_DIR}")

for variable_name in (
    "display", "masked_change_map", "change_map", "change_figure", "change_axes",
    "change_handles", "change_cmap", "change_labels",
    "change_harmonized_a", "change_harmonized_b", "change_valid_a", "change_valid_b",
    "change_source_a", "change_source_b", "change_valid", "change_mask_a", "change_mask_b",
    "change_both", "change_loss", "change_gain", "change_neither",
):
    globals().pop(variable_name, None)
gc.collect()

## Discusión final

Utiliza las tablas, mapas y gráficos de ambas partes para distinguir las diferencias entre fuentes de los cambios temporales de la clase seleccionada.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
area_by_source.to_csv(OUTPUT_DIR / "superficies_multiclase_por_fuente.csv", index=False)
area_comparison.to_csv(OUTPUT_DIR / "comparacion_superficies_multiclase.csv", index=False)
crosswalk.to_csv(OUTPUT_DIR / "correspondencias_espaciales_multiclase.csv", index=False)
crosswalk_pct.to_csv(OUTPUT_DIR / "correspondencias_espaciales_multiclase_porcentaje.csv")
print(f"Resultados multiclase guardados en: {OUTPUT_DIR}")